In [1]:
import os
import glob
import pandas as pd
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [8]:
# Dataset paths
AUDIO_DIR = "../ESC-50-master/audio/"
CSV_PATH = "../ESC-50-master/meta/esc50.csv"
REAL_MIC_DIR = "../ESC-50-master/real_mic_data/"


In [3]:
# Audio processing parameters - MUST match Flutter app exactly
SR = 22050
DURATION = 5
N_MELS = 64
MAX_PAD_LEN = 216

# Sound classes for deaf accessibility app
IMPORTANT_CLASSES = [
    'siren', 'crying_baby', 'door_wood_knock', 'glass_breaking',
    'fireworks', 'car_horn'
]

LABEL_MAP = {category: idx for idx, category in enumerate(IMPORTANT_CLASSES)}


In [4]:
# Convert raw audio to mel-spectrogram matching librosa defaults (n_fft=2048, hop_length=512)
def process_audio_to_mel(audio, sr, n_mels=N_MELS, max_pad_len=MAX_PAD_LEN):
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    if mel_spec_db.shape[1] < max_pad_len:
        pad_width = max_pad_len - mel_spec_db.shape[1]
        mel_spec_db = np.pad(mel_spec_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
    else:
        mel_spec_db = mel_spec_db[:, :max_pad_len]
    return mel_spec_db

In [5]:
# Augmentation: add synthetic white noise
def add_noise(data, noise_factor=0.005):
    noise = np.random.randn(len(data))
    return data + noise_factor * noise

# Augmentation: pitch shift the audio slightly
def pitch_shift(data, sr, n_steps=2):
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=n_steps)

In [6]:
# Load training data from ESC-50 dataset
print("Loading ESC-50 clips...")
raw_clips = []

df = pd.read_csv(CSV_PATH)
df_filtered = df[df['category'].isin(IMPORTANT_CLASSES)].reset_index(drop=True)

for _, row in df_filtered.iterrows():
    path = os.path.join(AUDIO_DIR, row['filename'])
    try:
        audio, sr = librosa.load(path, sr=SR, res_type='kaiser_fast')
        label = LABEL_MAP[row['category']]
        raw_clips.append((audio, sr, label))
    except Exception as e:
        print(f"Warning: {path} - {e}")

print(f"Loaded {len(raw_clips)} ESC-50 clips")

Loading ESC-50 clips...


c:\Users\ACER\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\ACER\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\ACER\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Loaded 240 ESC-50 clips


In [9]:
# Load real microphone recordings as held-out test set (never use for training/validation)
print("\nLoading real microphone clips (for held-out test set)...")
real_test_clips = []

for category in IMPORTANT_CLASSES:
    class_dir = os.path.join(REAL_MIC_DIR, category)
    if not os.path.isdir(class_dir):
        print(f"  No folder for '{category}' - skipping")
        continue
    
    files = glob.glob(os.path.join(class_dir, "*"))
    for path in files:
        try:
            audio, sr = librosa.load(path, sr=SR, res_type='kaiser_fast')
            target_len = SR * DURATION
            if len(audio) < target_len:
                audio = np.pad(audio, (0, target_len - len(audio)))
            else:
                audio = audio[:target_len]
            real_test_clips.append((audio, sr, LABEL_MAP[category]))
        except Exception as e:
            print(f"Warning: {path} - {e}")

print(f"Loaded {len(real_test_clips)} real microphone clips for testing")


Loading real microphone clips (for held-out test set)...
Loaded 30 real microphone clips for testing


In [10]:
labels_only = [c[2] for c in raw_clips]
train_idx, val_idx = train_test_split(
    range(len(raw_clips)),
    test_size=0.2,
    random_state=42,
    stratify=labels_only,
)

train_raw = [raw_clips[i] for i in train_idx]
val_raw = [raw_clips[i] for i in val_idx]

print(f"\nESC-50 split: Train {len(train_raw)} | Val {len(val_raw)}")

def build_augmented_set(raw_set, augment: bool):
    X, y = [], []
    for audio, sr, label in raw_set:
        X.append(process_audio_to_mel(audio, sr))
        y.append(label)
        if augment:
            X.append(process_audio_to_mel(add_noise(audio), sr))
            y.append(label)
            X.append(process_audio_to_mel(pitch_shift(audio, sr, n_steps=2), sr))
            y.append(label)
    return np.array(X), np.array(y)

print("Building augmented training set...")
X_train, y_train = build_augmented_set(train_raw, augment=True)

print("Building validation set (no augmentation)...")
X_val, y_val = build_augmented_set(val_raw, augment=False)

print("Building real-world test set (held-out, no augmentation)...")
X_real_test, y_real_test = build_augmented_set(real_test_clips, augment=False)

X_train = X_train.reshape(-1, X_train.shape[1], X_train.shape[2], 1)
X_val = X_val.reshape(-1, X_val.shape[1], X_val.shape[2], 1)
X_real_test = X_real_test.reshape(-1, X_real_test.shape[1], X_real_test.shape[2], 1)

print(f"\nTrain shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Real-world test shape: {X_real_test.shape}")


ESC-50 split: Train 192 | Val 48
Building augmented training set...
Building validation set (no augmentation)...
Building real-world test set (held-out, no augmentation)...

Train shape: (576, 64, 216, 1)
Val shape: (48, 64, 216, 1)
Real-world test shape: (30, 64, 216, 1)


In [11]:
model = models.Sequential([
    layers.Input(shape=(N_MELS, MAX_PAD_LEN, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(len(IMPORTANT_CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True
)

print("\nTraining...")
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=[early_stopper],
)

y_val_pred = np.argmax(model.predict(X_val), axis=1)



Training...
Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 102ms/step - accuracy: 0.3905 - loss: 6.8338 - val_accuracy: 0.2917 - val_loss: 19.1884
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - accuracy: 0.6762 - loss: 1.4607 - val_accuracy: 0.3333 - val_loss: 21.6351
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.8191 - loss: 0.6886 - val_accuracy: 0.6042 - val_loss: 5.0786
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - accuracy: 0.8325 - loss: 0.6660 - val_accuracy: 0.4375 - val_loss: 16.7255
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - accuracy: 0.8169 - loss: 0.7224 - val_accuracy: 0.5833 - val_loss: 5.6368
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - accuracy: 0.9304 - loss: 0.2081 - val_accuracy: 0.6667 - val_loss: 3.3202
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 116ms/step - accuracy: 0.8868 - loss: 0.4194 - val_accuracy: 0.6667 - val_loss: 2.7467
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.9060 - loss: 0.4178

In [12]:
print("\n" + "="*60)
print("VALIDATION SET PERFORMANCE (ESC-50 Clean Data)")
print("="*60)
print(classification_report(y_val, y_val_pred, target_names=IMPORTANT_CLASSES))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

if len(y_real_test) > 0:
    y_real_pred = np.argmax(model.predict(X_real_test), axis=1)
    
    print("\n" + "="*60)
    print("REAL-WORLD TEST PERFORMANCE (Phone Microphone Recording)")
    print("="*60)
    print(f"(n={len(y_real_test)}, ~5 per class)")
    print(classification_report(y_real_test, y_real_pred, target_names=IMPORTANT_CLASSES, zero_division=0))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_real_test, y_real_pred))
else:
    print("\nNo real-world test clips loaded.")

print("\nExporting to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("emergency_audio_classifier.tflite", "wb") as f:
    f.write(tflite_model)

with open("labels.txt", "w") as f:
    for category in IMPORTANT_CLASSES:
        f.write(f"{category}\n")

print("Saved: emergency_audio_classifier.tflite")
print("Saved: labels.txt")


VALIDATION SET PERFORMANCE (ESC-50 Clean Data)
                 precision    recall  f1-score   support

          siren       0.75      0.75      0.75         8
    crying_baby       0.73      1.00      0.84         8
door_wood_knock       0.54      0.88      0.67         8
 glass_breaking       0.71      0.62      0.67         8
      fireworks       1.00      0.12      0.22         8
       car_horn       0.62      0.62      0.62         8

       accuracy                           0.67        48
      macro avg       0.73      0.67      0.63        48
   weighted avg       0.73      0.67      0.63        48


Confusion Matrix:
[[6 2 0 0 0 0]
 [0 8 0 0 0 0]
 [0 0 7 1 0 0]
 [0 0 2 5 0 1]
 [1 0 4 0 1 2]
 [1 1 0 1 0 5]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step

REAL-WORLD TEST PERFORMANCE (Phone Microphone Recording)
(n=30, ~5 per class)
                 precision    recall  f1-score   support

          siren       0.21      0.60      0.32         5
    crying_baby       0.67      0.40 

INFO:tensorflow:Assets written to: C:\Users\ACER\AppData\Local\Temp\tmpryh0gojv\assets


Saved artifact at 'C:\Users\ACER\AppData\Local\Temp\tmpryh0gojv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 216, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1783108295504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108296464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108296080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108297424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108296656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108297232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108298576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108299152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108299344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108298192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1783108